In [ ]:

# Colab Cell 1 — Install dependencies
# Run this cell first. It may take a minute.

!pip install --upgrade pip >/dev/null
!pip install azure-storage-blob mysql-connector-python requests tqdm python-dotenv Pillow >/dev/null


In [ ]:
# Colab Cell 2 — MySQL connection and table setup
# Configure via environment variables OR fill the config dict below.
import os
from datetime import datetime
import mysql.connector
from mysql.connector import errorcode

# Option A: environment variables (recommended)
MYSQL_HOST = os.getenv("MYSQL_HOST", "cycleai.mysql.database.azure.com")            # e.g. "127.0.0.1" or cloud host
MYSQL_PORT = int(os.getenv("MYSQL_PORT", "3306"))
MYSQL_USER = os.getenv("MYSQL_USER", "admindev")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD", "Zana81333")
MYSQL_DB = os.getenv("MYSQL_DB", "cycle_ai")

# Connect helper (returns connection)
def get_mysql_conn():
    cfg = {
        "host": MYSQL_HOST,
        "port": MYSQL_PORT,
        "user": MYSQL_USER,
        "password": MYSQL_PASSWORD,
        "database": MYSQL_DB,
        "autocommit": False,
    }
    try:
        conn = mysql.connector.connect(**cfg)
        return conn
    except mysql.connector.Error as err:
        if err.errno == errorcode.ER_ACCESS_DENIED_ERROR:
            print("Something is wrong with your user name or password")
        elif err.errno == errorcode.ER_BAD_DB_ERROR:
            print("Database does not exist")
            raise

print("MySQL connected and table ensured: perception_poll_votes_ai")


In [ ]:
# Colab Cell 3 — Azure Blob connection and caching blob names
import os, io, random
from azure.storage.blob import BlobServiceClient
from PIL import Image

# Configure via environment variable or fill inline
AZURE_CONN_STR = os.getenv("AZURE_CONN_STR", "")   # recommended: store in Colab secret or env
BLOB_CONTAINER = os.getenv("BLOB_CONTAINER", "london")

# Fallback inline example:
# AZURE_CONN_STR = "DefaultEndpointsProtocol=...;AccountName=...;AccountKey=...;EndpointSuffix=core.windows.net"
# BLOB_CONTAINER = "poll-images"

if not AZURE_CONN_STR:
    raise RuntimeError("AZURE_CONN_STR not set. Set as environment variable or inline in this cell.")

blob_svc = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
container_client = blob_svc.get_container_client(BLOB_CONTAINER)

print("Listing blobs in 'Images/' folder (this may take a few seconds)...")

IMAGE_PREFIX = "training/training/Images/"
all_image_blobs = container_client.list_blobs(name_starts_with=IMAGE_PREFIX)

IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.webp')

blob_names = [
    blob.name for blob in all_image_blobs
    if blob.name.lower().endswith(IMAGE_EXTENSIONS)
]

print(f"Found {len(blob_names)} image files.")
if len(blob_names) < 2:
    raise RuntimeError(f"Need at least 2 blobs in container to sample pairs. Found: {len(blob_names)}")

def download_blob_to_pil(blob_name, timeout=30):
    """Download a blob and return it as a PIL Image."""
    blob_data = container_client.download_blob(blob_name, timeout=timeout).readall()
    img = Image.open(io.BytesIO(blob_data)).convert("RGB")
    return img

def sample_two_random_blobs():
    a, b = random.sample(blob_names, 2)
    return a, b

print("Azure blob helpers ready.")


In [ ]:
# Force-uninstall the potentially broken/old package
!pip uninstall -y transformers

# Install the LATEST (development) version from GitHub and dependencies
!pip install --upgrade git+https://github.com/huggingface/transformers.git
!pip install --upgrade accelerate einops sentencepiece transformers_stream_generator

print("\n\n--- Installation complete! ---")
print(">>> PLEASE RESTART YOUR RUNTIME NOW. <<<")
# Go to the menu: Runtime > Restart session

In [ ]:
# 2. Run this cell AFTER restarting your runtime
import torch
import requests
from PIL import Image
from io import BytesIO
from transformers import AutoTokenizer, Qwen2VLForConditionalGeneration # Note the change here

# SETUP
HF_MODEL = "Qwen/Qwen2-VL-7B-Instruct"

# Determine device and dtype
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

print(f"Loading model: {HF_MODEL} on {device} with {dtype}")

# 3. LOAD MODEL: ***CRITICAL FIX: Use Qwen2VLForConditionalGeneration***
model = Qwen2VLForConditionalGeneration.from_pretrained(
    HF_MODEL,
    torch_dtype=dtype,  # Corrected argument
    device_map="auto",
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL, trust_remote_code=True)

print("Model and tokenizer loaded successfully.")



In [ ]:
# Colab Cell 4 - Qwen2-VL

# ---------- LIBRARY IMPORTS ----------
import os, io, json, time, base64, hashlib, random, re
from tqdm import tqdm
from PIL import Image
import requests
import mysql.connector
from datetime import datetime
from time import sleep
import torch
import numpy as np
from IPython.display import display, HTML
from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, AutoImageProcessor
import traceback
import pandas as pd

# ---------- CONFIG ----------
HF_API_TOKEN = os.getenv("HF_TOKEN", "")
WORKER_ID = os.getenv("WORKER_ID", "colab_worker_2")
TARGET_SIZE = (640, 640)

N = 3
BATCH_COMMIT = 20
RATE_LIMIT_SECONDS = float(os.getenv("RATE_LIMIT_SECONDS", "0.1"))
RETRY_MAX = 5
SAMPLE_TIMEOUT = 30
DEFAULT_CONFIDENCE_ON_FAIL = 0.0

vote_counts = {0: 0, 1: 0, 2: 0}
results = []

FULL_PROMPT_TEXT = (
    "You will see two images: Image A (first) and Image B (second).\n\n"
    "TASK: Decide which image looks SAFER for cyclists.\n\n"
    "Respond with ONE line only in the exact required format:\n"
    "[VOTE]: [Vote Number] - [short reason]\n\n"
    "Where Vote Number is:\n"
    "  0 - Both look equally safe\n"
    "  1 - Image A looks safer\n"
    "  2 - Image B looks safer\n\n"
    "Keep your reason short. DO NOT include any extra text.\n\n"
    "Examples (Use this format EXACTLY):\n"
    "[VOTE]: 1 - Image A has a dedicated bike lane.\n"
    "[VOTE]: 2 - Image B has fewer vehicles present.\n"
    "[VOTE]: 0 - Both show similar traffic conditions.\n"
    "\nNow decide which image looks safer."
)



USER_PROMPT_FINAL = FULL_PROMPT_TEXT

def pil_to_data_uri(pil_img, quality=85):
    b64 = pil_to_base64_jpeg(pil_img, quality=quality)
    return f"data:image/jpeg;base64,{b64}"

def display_pair(pil1, pil2, vote, reason, name1, name2, max_width=320):
    uri1 = pil_to_data_uri(pil1)
    uri2 = pil_to_data_uri(pil2)
    caption = f"<b>Vote:</b> {vote} &nbsp;&nbsp; <b>Reason:</b> {reason}<br><b>Files:</b> {name1} — {name2}"
    html = f"""
    <div style="display:flex; align-items:flex-start; gap:12px; margin-bottom:10px;">
      <div style="text-align:center;">
        <img src="{uri1}" style="max-width:{max_width}px; height:auto; display:block; border:1px solid #ccc;" /><br/>
        <small>{name1}</small>
      </div>
      <div style="text-align:center;">
        <img src="{uri2}" style="max-width:{max_width}px; height:auto; display:block; border:1px solid #ccc;" /><br/>
        <small>{name2}</small>
      </div>
      <div style="flex:1; padding-left:8px;">
        {caption}
      </div>
    </div>
    """
    display(HTML(html))

def preprocess_pil_to_tensor(pil_img, size=TARGET_SIZE):
    img = pil_img.convert("RGB").resize(size, resample=Image.BICUBIC)
    arr = np.asarray(img).astype(np.float32) / 255.0
    tensor = torch.from_numpy(arr).permute(2, 0, 1)
    return tensor

# ---------- UTILITIES ----------
def pil_to_base64_jpeg(pil_img, quality=85):
    buff = io.BytesIO()
    pil_img.save(buff, format="JPEG", quality=quality)
    return base64.b64encode(buff.getvalue()).decode("utf-8")

def idempotency_key(name1, name2):
    s = f"{name1}|{name2}"
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def upsert_vote(conn, rec):
    cur = conn.cursor()
    sql = """
    INSERT INTO perception_poll_votes_ai
      (image_name1, image_name2, vote, model_id, user_id, reason)
    VALUES (%s, %s, %s, %s, %s, %s)
    """
    params = (rec["image_name1"], rec["image_name2"], rec["vote"], rec["model_id"], rec["user_id"], rec["reason"])
    cur.execute(sql, params)
    cur.close()

def debug_tokenizer_model(tokenizer, model, image_processor=None):
    """
    Lightweight diagnostics to paste into chat if things still don't work.
    Prints method presence and a couple of small sample transformations.
    """
    info = {}
    info["tokenizer_type"] = str(type(tokenizer))
    info["tokenizer_has_from_list_format"] = hasattr(tokenizer, "from_list_format")
    info["tokenizer_attrs_sample"] = sorted([a for a in dir(tokenizer) if not a.startswith("_")])[:80]
    info["model_type"] = str(type(model))
    try:
        info["model_config_keys"] = list(model.config.to_diff_dict().keys())
    except Exception:
        info["model_config"] = str(getattr(model, "config", "no-config"))
    if image_processor is None:
        try:
            image_processor = AutoImageProcessor.from_pretrained(getattr(model, "name_or_path", HF_MODEL), trust_remote_code=True)
            info["image_processor_loaded"] = True
            info["image_processor_keys"] = list(image_processor.__dict__.keys())[:40]
        except Exception as e:
            info["image_processor_loaded"] = False
            info["image_processor_err"] = repr(e)
    else:
        info["image_processor_present"] = True
    print(json.dumps(info, indent=2, default=str))

def call_model_locally(user_prompt_text, pil_list, model, tokenizer, max_new_tokens=128):
    """
    Calls Qwen2-VL with two PIL images and a user prompt.
    Returns the raw decoded string from the model (not parsed).
    """
    device = model.device
    model_name = getattr(model, "name_or_path", None) or globals().get("HF_MODEL")
    if not model_name:
        raise RuntimeError("Could not determine model name for image processor.")

    try:
        from transformers import AutoProcessor
        processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)

        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": pil_list[0]},
                    {"type": "image", "image": pil_list[1]},
                    {"type": "text", "text": user_prompt_text},
                ],
            }
        ]

        inputs = processor.apply_chat_template(
            conversation,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        )

        # Move tensors to device & correct dtype
        for k, v in inputs.items():
            if isinstance(v, torch.Tensor):
                if v.dtype in (torch.long, torch.int64, torch.int32, torch.int):
                    inputs[k] = v.to(device)
                else:
                    inputs[k] = v.to(device, dtype=model.dtype)

        # Generate (longer max_new_tokens to avoid truncation)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                return_dict_in_generate=True,
                output_scores=False
            )

        # Extract sequences robustly
        sequences = getattr(outputs, "sequences", None)
        if sequences is None and isinstance(outputs, dict):
            sequences = outputs.get("sequences", None)
        if sequences is None:
            # fallback: try to use outputs[0] if it's a list/tuple
            sequences = outputs[0] if isinstance(outputs, (list, tuple)) and len(outputs) > 0 else outputs

        if isinstance(sequences, torch.Tensor):
            seq = sequences[0] if sequences.dim() > 1 else sequences
        elif isinstance(sequences, (list, tuple)):
            seq = torch.tensor(sequences[0]) if len(sequences) > 0 else torch.tensor(sequences)
        else:
            seq = torch.tensor(sequences)

        input_len = inputs["input_ids"].shape[1] if "input_ids" in inputs else 0
        gen_ids = seq[input_len:] if seq.shape[0] > input_len else seq

        gen_id_list = gen_ids.cpu().tolist() if isinstance(gen_ids, torch.Tensor) else list(gen_ids)
        if not gen_id_list:
            raise RuntimeError("Generated id list is empty.")

        return tokenizer.decode(gen_id_list, skip_special_tokens=True).strip()

    except Exception as e:
        raise RuntimeError(f"call_model_locally failed: {e}")



def parse_vote_reason(txt):
    txt = (txt or "").strip()
    match = re.match(r"^\[VOTE\]:?\s*([0-2])\s*-\s*(.+)?$", txt, flags=re.S)
    if not match:
        return None, None
    vote = int(match.group(1))
    reason = (match.group(2) or "").strip()
    return vote, reason

def reason_is_incomplete(reason):
    if not reason:
        return True
    words = reason.split()
    if len(words) < 3:  # require at least 3 words for your ≤6 words prompt
        return True
    incomplete_endings = ("because", "because it", "because it has", "because it is",
                          "because the", "because they", "because there")
    low = reason.lower().strip()
    if any(low.endswith(s) for s in incomplete_endings):
        return True
    if low[-1] not in (".", "!", "?"):
        return True
    return False


# ---------- MAIN LOOP (drop-in replacement) ----------
print(f"Starting main loop: N={N}, HF model={HF_MODEL}, worker={WORKER_ID}")
conn = get_mysql_conn()
insert_count = 0

try:
    blob_names
except NameError:
    from azure.storage.blob import BlobServiceClient
    IMAGE_PREFIX = "training/training/Images/"
    IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.webp')

    print(f"Connecting to Azure to list files starting with '{IMAGE_PREFIX}'...")

    blob_svc = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
    container_client = blob_svc.get_container_client(BLOB_CONTAINER)

    all_image_blobs = container_client.list_blobs(name_starts_with=IMAGE_PREFIX)

    blob_names = [
        blob.name for blob in all_image_blobs
        if blob.name.lower().endswith(IMAGE_EXTENSIONS)
    ]

    if not blob_names:
        raise RuntimeError(f"No image files found in the '{IMAGE_PREFIX}' directory of the container.")

    print(f"Found {len(blob_names)} image files to sample from.")

pbar = tqdm(range(N), desc="Model votes")
for i in pbar:
    # sample two different files
    name1, name2 = random.sample(blob_names, 2)

    # fetch blobs with exponential backoff
    pil1 = pil2 = None
    for attempt in range(RETRY_MAX):
        try:
            pil1 = download_blob_to_pil(name1)
            pil2 = download_blob_to_pil(name2)
            break
        except Exception as e:
            wait = 2 ** attempt
            print(f"Blob download error for pair ({name1},{name2}) attempt {attempt+1}: {e}; sleeping {wait}s")
            time.sleep(wait)
    if pil1 is None or pil2 is None:
        pbar.set_postfix(status="download_fail")
        continue

    # call model (pass list of PILs)
    try:
        raw = call_model_locally(FULL_PROMPT_TEXT, [pil1, pil2], model, tokenizer, max_new_tokens=128)
    except Exception as e:
        print("Model call failed, skipping:", repr(e))
        pbar.set_postfix(status="model_fail")
        continue

    txt = str(raw) if raw is not None else ""
    # Audit print
    print("-" * 60)
    print(f"Model Output for ({name1.split('/')[-1]}, {name2.split('/')[-1]}):")
    print(txt)
    print("-" * 60)

    # parse robustly: accept formats like:
    # "[VOTE]: 2 - reason", "2 - reason", "2: reason", "2 | reason", "2" etc.
    match = re.search(r"^(?:\[VOTE\]\s*[:\-]?\s*)?\s*([0-2])\s*(?:[:\-|\|])\s*(.*)$", txt.strip(), re.MULTILINE | re.IGNORECASE)
    if not match:
        # fallback: single digit anywhere at start and the rest is reason
        match2 = re.search(r"^\s*([0-2])\b(.*)$", txt.strip(), re.MULTILINE)
        if not match2:
            print(f"No valid 0/1/2 vote parsed for pair ({name1}, {name2}). Raw model text (truncated): {txt[:200]}")
            pbar.set_postfix(status="parse_fail")
            continue
        vote_val = int(match2.group(1))
        reason_text = match2.group(2).strip()
    else:
        vote_val = int(match.group(1))
        reason_text = (match.group(2) or "").strip()

    # normalize reason_text to a short extraction (avoid enormous strings)
    if reason_text:
        # trim to one line and 200 chars max for DB
        reason_text = reason_text.splitlines()[0].strip()[:200]
    else:
        reason_text = "No reasoning provided."

    # Clean blob names for DB
    clean_name1 = name1.replace(IMAGE_PREFIX, '', 1)
    clean_name2 = name2.replace(IMAGE_PREFIX, '', 1)

    rec = {
        "id": idempotency_key(clean_name1, clean_name2),
        "image_name1": clean_name1,
        "image_name2": clean_name2,
        "vote": int(vote_val),
        "model_id": HF_MODEL,
        "user_id": WORKER_ID,
        "reason": reason_text,
    }

    try:
        display_pair(pil1, pil2, vote_val, reason_text, clean_name1.split('/')[-1], clean_name2.split('/')[-1])
    except Exception as e:
        print("Display failed:", e)

    # insert/update DB (upsert_vote should not commit itself in this flow)
    try:
        upsert_vote(conn, rec)
        insert_count += 1
    except Exception as e:
        print("DB insert failed for record:", repr(e))
        try:
            conn.rollback()
        except Exception:
            pass
        pbar.set_postfix(status="db_error")
        continue
    vote_counts[vote_val] = vote_counts.get(vote_val, 0) + 1
    results.append({
      "image1": clean_name1,
      "image2": clean_name2,
      "vote": vote_val,
      "reason": reason_text,
      "model_id": HF_MODEL,
      "user_id": WORKER_ID
    })

    # periodic commit
    if insert_count % BATCH_COMMIT == 0:
        try:
            conn.commit()
        except Exception as e:
            print("DB commit failed:", repr(e))
            try:
                conn.rollback()
            except Exception:
                pass

    pbar.set_postfix(last_vote=rec["vote"])
    time.sleep(RATE_LIMIT_SECONDS)

# final commit + close
try:
    conn.commit()
except Exception:
    try:
        conn.rollback()
    except Exception:
        pass
finally:
    try:
        conn.close()
    except Exception:
        pass

print("Done. Inserted/updated records:", insert_count)

print("\nVote totals (during this run):")
for k in sorted(vote_counts.keys()):
    print(f"  Vote {k}: {vote_counts[k]}")

if results:
    df = pd.DataFrame(results)
    display(df.head(50))
else:
    print("No successful results captured in 'results' list.")
